# Pass 2 — Consolidate & Deduplicate Objectives

**Input:** Pass 1 output (per-column extractions for each fund).  
**Task:** The LLM sees ALL per-column extractions for one fund and:  
1. Matches equivalent objectives across languages/columns  
2. Deduplicates (same objective stated in English, French, German = one objective)  
3. Produces a final consolidated list with English text and classification  

**Output:** One row per fund with final deduplicated objectives.

In [61]:
import pandas as pd
from tqdm import tqdm
import time, os, json, anthropic
from pathlib import Path

with open("Claude_API.txt", "r") as file:
    api_key = file.read().strip()
os.environ['ANTHROPIC_API_KEY'] = api_key

config = {}
with open("File_Directory.txt", "r") as file:
    for line in file:
        if ":" in line:
            key, value = line.split(":", 1)
            config[key.strip()] = value.strip()

OUTPUT_DIR = Path(config["Output"])
MODEL = "claude-sonnet-4-6"  # UPDATE as needed

In [70]:
# === LOAD PASS 1 OUTPUT ===
# UPDATE this path to your actual Pass 1 output file
PASS1_FILE = os.path.join(OUTPUT_DIR, "Pass1_Extract_22_funds_20260608_1456.xlsx")  # UPDATE

p1_df = pd.read_excel(PASS1_FILE)
p1_df['pass1_raw'] = p1_df['pass1_raw'].apply(json.loads)
p1_df['columns_sent'] = p1_df['columns_sent'].apply(json.loads)
print(f"Loaded {len(p1_df)} funds from Pass 1")

Loaded 22 funds from Pass 1


In [71]:
PASS2_SYSTEM_PROMPT = """You are consolidating fund objective extractions that were made independently from multiple regulatory text columns for the same European mutual fund.

You will receive a JSON object where each key is a column name, and the value contains:
- "language": the language of that column
- "objectives": a list of objectives extracted from that column, each with:
  - "objective_text": concise English summary of the goal
  - "source_text": verbatim excerpt from the source in original language (audit trail)
  - "objective_type": "financial", "sustainable", or "sustainable_disclosure"
    (sustainable_disclosure = a sustainability clause that uses regulatory language
    and may be a mandatory disclosure rather than a fund-chosen objective — carry
    this classification through unchanged; do not promote it to "sustainable")

YOUR TASK:
1. MATCH equivalent objectives across columns/languages. The same objective may appear in English, French, German, Swedish, etc. Compare objective_text values — since these are clean English summaries, matching across languages is straightforward.
2. DEDUPLICATE: If multiple columns express the same objective, keep it only once.
3. For each unique objective, select the BEST objective_text phrasing — prefer a native English source if available; otherwise use or improve the translation. Keep the source_text from the most authoritative source column.
4. Classify each as "financial", "sustainable", or "sustainable_disclosure".
5. Record which columns contained this objective (for traceability).

MATCHING GUIDANCE:
- "long-term capital growth" and "achieve long-term capital growth" = SAME objective
- "outperform the benchmark" and "exceed the benchmark index" = SAME objective (minor wording variation)
- "achieve capital growth" and "achieve capital growth" plus "outperform the benchmark" — the second set contains TWO objectives; match the first and keep the second as separate
- Be generous in matching across languages but strict about not merging genuinely different objectives
- Do NOT recombine objectives that were correctly split in Pass 1. If one column produced a single combined entry while another correctly produced two separate entries, the split form takes precedence.

OUTPUT FORMAT:
{
  "consolidated_objectives": [
    {
      "objective_number": 1,
      "objective_text": "the final concise English statement of this objective",
      "source_text": "verbatim excerpt from the most authoritative source column",
      "objective_type": "financial" or "sustainable" or "sustainable_disclosure",
      "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", ...],
      "match_notes": "brief note on how columns were matched, or null if only found in one column"
    }
  ],
  "consolidation_notes": "any important notes about the consolidation process"
}

If Pass 1 found NO objectives in ANY column:
{
  "consolidated_objectives": [],
  "consolidation_notes": "NOT IDENTIFIED — no objectives found in any column"
}
"""

In [72]:
PASS2_FEW_SHOT = [
    {
        "fund_name": "Example Multilingual Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital et \u00e0 surperformer l'indice de r\u00e9f\u00e9rence.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark index", "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital et \u00e0 surperformer l'indice de r\u00e9f\u00e9rence.", "objective_type": "financial"}
                ]
            },
            "PRIIPS KID Objective - German": {
                "language": "German",
                "objectives": [
                    {"objective_text": "achieve capital growth", "source_text": "Der Fonds strebt Kapitalwachstum an und versucht, die Benchmark zu \u00fcbertreffen.", "objective_type": "financial"},
                    {"objective_text": "outperform the benchmark", "source_text": "Der Fonds strebt Kapitalwachstum an und versucht, die Benchmark zu \u00fcbertreffen.", "objective_type": "financial"},
                    {"objective_text": "reduce greenhouse gas emissions", "source_text": "Nachhaltiges Anlageziel des Fonds ist die Reduzierung der Treibhausgasemissionen.", "objective_type": "sustainable"}
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve capital growth",
                    "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same objective across all three language columns"
                },
                {
                    "objective_number": 2,
                    "objective_text": "outperform the benchmark",
                    "source_text": "The fund seeks to achieve capital growth and to outperform the benchmark.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French", "PRIIPS KID Objective - German"],
                    "match_notes": "Same benchmark-beating objective across all three languages"
                },
                {
                    "objective_number": 3,
                    "objective_text": "reduce greenhouse gas emissions",
                    "source_text": "Nachhaltiges Anlageziel des Fonds ist die Reduzierung der Treibhausgasemissionen.",
                    "objective_type": "sustainable",
                    "found_in_columns": ["PRIIPS KID Objective - German"],
                    "match_notes": "Sustainability objective found only in German column"
                }
            ],
            "consolidation_notes": "Two financial objectives matched across all three languages. One sustainability objective found only in the German column."
        }
    },
    {
        "fund_name": "Example Sustainable Disclosure Fund",
        "pass1_data": {
            "PRIIPS KID Objective": {
                "language": "English",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "The fund seeks to achieve long-term capital growth.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            },
            "PRIIPS KID Objective - French": {
                "language": "French",
                "objectives": [
                    {
                        "objective_text": "achieve long-term capital growth",
                        "source_text": "Le fonds cherche \u00e0 r\u00e9aliser une croissance du capital \u00e0 long terme.",
                        "objective_type": "financial"
                    },
                    {
                        "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                        "source_text": "Le Sous-Fonds investit au moins 50% de ses actifs dans des Investissements durables, tels que d\u00e9finis par le SFDR.",
                        "objective_type": "sustainable_disclosure"
                    }
                ]
            }
        },
        "response": {
            "consolidated_objectives": [
                {
                    "objective_number": 1,
                    "objective_text": "achieve long-term capital growth",
                    "source_text": "The fund seeks to achieve long-term capital growth.",
                    "objective_type": "financial",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French"],
                    "match_notes": "Same financial objective in English and French columns"
                },
                {
                    "objective_number": 2,
                    "objective_text": "invest at least 50% of assets in Sustainable Investments, as defined under SFDR",
                    "source_text": "The Sub-Fund invests at least 50% of assets in Sustainable Investments, as defined under SFDR.",
                    "objective_type": "sustainable_disclosure",
                    "found_in_columns": ["PRIIPS KID Objective", "PRIIPS KID Objective - French"],
                    "match_notes": "Same regulatory disclosure in both languages; classification carried through as sustainable_disclosure unchanged"
                }
            ],
            "consolidation_notes": "One financial objective and one sustainable_disclosure item. The sustainable_disclosure classification is preserved from Pass 1 for human review."
        }
    }
]

In [65]:
import re

def robust_json_parse(text):
    if not isinstance(text, str):
        return None
    cleaned = text.strip()
    
    # 1. Strip markdown fences
    if cleaned.startswith("```"):
        cleaned = re.sub(r'^```\w*\n?', '', cleaned)
        cleaned = re.sub(r'\n?```\s*$', '', cleaned)
        cleaned = cleaned.strip()
    
    # 2. Replace smart quotes with unicode escapes (always, not just after fences)
    cleaned = cleaned.replace('„', '\\u201E')
    cleaned = cleaned.replace('\u201c', '\\u201C')
    cleaned = cleaned.replace('\u201d', '\\u201D')
    cleaned = cleaned.replace('«', '\\u00AB')
    cleaned = cleaned.replace('»', '\\u00BB')
    cleaned = cleaned.replace('‚', '\\u201A')
    cleaned = cleaned.replace('\u2018', '\\u2018')
    cleaned = cleaned.replace('\u2019', '\\u2019')
    
    # 3. Try direct parse
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # 4. Fix unescaped control characters
    def fix_strings(match):
        s = match.group(0)
        s = s.replace('\n', '\\n')
        s = s.replace('\r', '\\r')
        s = s.replace('\t', '\\t')
        return s
    fixed = re.sub(r'"(?:[^"\\]|\\.)*"', fix_strings, cleaned, flags=re.DOTALL)
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass

    # 5. Last resort — extract outermost { }
    brace_match = re.search(r'\{.*\}', fixed, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass

    return None

In [66]:
def pass2_consolidate(fund_name, fund_id, pass1_data):
    """Consolidate per-column extractions into deduplicated objectives."""
    # Filter to only columns that had objectives
    cols_with_data = {}
    for col, data in pass1_data.items():
        if col.startswith('_'):
            continue
        if isinstance(data, dict) and 'objectives' in data and len(data['objectives']) > 0:
            cols_with_data[col] = data

    if not cols_with_data:
        return {
            "consolidated_objectives": [],
            "consolidation_notes": "NOT IDENTIFIED — Pass 1 found no objectives in any column"
        }

    user_prompt = f"""Fund ID: {fund_id}
Fund Name: {fund_name}

Pass 1 extractions (per-column):
{json.dumps(cols_with_data, indent=2)}"""

    messages = []
    for ex in PASS2_FEW_SHOT:
        messages.append({
            "role": "user",
            "content": f"Fund Name: {ex['fund_name']}\n\nPass 1 extractions (per-column):\n{json.dumps(ex['pass1_data'], indent=2)}"
        })
        messages.append({
            "role": "assistant",
            "content": json.dumps(ex["response"], indent=2)
        })

    messages.append({"role": "user", "content": user_prompt})

    try:
        client = anthropic.Anthropic()
        response = client.messages.create(
            model=MODEL,
            max_tokens=2000,
            temperature=0,
            system=PASS2_SYSTEM_PROMPT,
            messages=messages
        )
        print(f"   [{fund_name}] tokens — in: {response.usage.input_tokens}, out: {response.usage.output_tokens}")

        text = response.content[0].text
        parsed = robust_json_parse(text)
        if parsed is not None:
            return parsed
        return {"_error": f"JSON parse error after all attempts: {text[:300]}"}

    except Exception as e:
        print(f"   Error for {fund_name}: {e}")
        return {"_error": str(e)}

In [73]:
# === RUN PASS 2 ===
pass2_results = []

for idx in tqdm(range(len(p1_df)), desc="Pass 2 — Consolidate"):
    row = p1_df.iloc[idx]
    fund_id = row['FundId']
    fund_name = row['Fund_Name']
    pass1_data = row['pass1_raw']

    # Skip funds that errored in Pass 1
    if '_error' in pass1_data:
        pass2_results.append({
            'FundId': fund_id,
            'Fund_Name': fund_name,
            'pass2_raw': {'_error': f"Skipped — Pass 1 error: {pass1_data['_error']}"}
        })
        continue

    result = pass2_consolidate(fund_name, fund_id, pass1_data)

    pass2_results.append({
        'FundId': fund_id,
        'Fund_Name': fund_name,
        'pass2_raw': result
    })

    if idx > 0 and idx % 50 == 0:
        time.sleep(0.5)

pass2_df = pd.DataFrame(pass2_results)
print(f"\nPass 2 complete: {len(pass2_df)} funds processed")

Pass 2 — Consolidate:   5%|▍         | 1/22 [00:10<03:32, 10.10s/it]

   [BlackRock Sysmc Eq Fac Pl D EUR H Acc] tokens — in: 3194, out: 619


Pass 2 — Consolidate:   9%|▉         | 2/22 [00:16<02:35,  7.75s/it]

   [Cardif BNPP IP Smid Cap Euro] tokens — in: 2917, out: 434


Pass 2 — Consolidate:  14%|█▎        | 3/22 [00:21<02:09,  6.81s/it]

...

   [Partners Group Direct Eq II Eltif I(USD)] tokens — in: 3711, out: 323


Pass 2 — Consolidate:  95%|█████████▌| 21/22 [03:08<00:08,  8.46s/it]

   [Redwheel Global Intrinsic Val I GBP Acc] tokens — in: 2800, out: 215


Pass 2 — Consolidate: 100%|██████████| 22/22 [03:17<00:00,  8.98s/it]

   [UBS (Lux) Eq Fd EM Sst Ldrs (USD) P] tokens — in: 3297, out: 443

Pass 2 complete: 22 funds processed


In [76]:
# === FLATTEN FOR INSPECTION ===
flat_rows = []
for _, row in pass2_df.iterrows():
    raw = row['pass2_raw']
    base = {'FundId': row['FundId'], 'Fund_Name': row['Fund_Name']}

    if '_error' in raw:
        base['Number_of_Objectives'] = 0
        base['Number_Financial'] = 0
        base['Number_Sustainable'] = 0
        base['Consolidation_Notes'] = raw['_error']
        flat_rows.append(base)
        continue

    objs = raw.get('consolidated_objectives', [])
    base['Number_of_Objectives'] = len(objs)
    base['Number_Financial'] = sum(1 for o in objs if o.get('objective_type') == 'financial')
    base['Number_Sustainable'] = sum(1 for o in objs if o.get('objective_type') == 'sustainable')
    base['Consolidation_Notes'] = raw.get('consolidation_notes', '')

    for i in range(5):  # support up to 5 objectives
        if i < len(objs):
            o = objs[i]
            base[f'Objective_{i+1}'] = o.get('objective_text', '')
            base[f'Objective_{i+1}_Source'] = o.get('source_text', '')
            base[f'Objective_{i+1}_Type'] = o.get('objective_type', '')
            base[f'Objective_{i+1}_Columns'] = ', '.join(o.get('found_in_columns', []))
        else:
            base[f'Objective_{i+1}'] = None
            base[f'Objective_{i+1}_Type'] = None
            base[f'Objective_{i+1}_Columns'] = None

    flat_rows.append(base)

pass2_flat = pd.DataFrame(flat_rows)

print("PASS 2 SUMMARY:")
print(f"  Funds: {len(pass2_flat)}")
print(f"  With ≥1 objective: {(pass2_flat['Number_of_Objectives'] > 0).sum()}")
print(f"  Avg objectives: {pass2_flat['Number_of_Objectives'].mean():.1f}")
print(f"\nObjective count distribution:")
print(pass2_flat['Number_of_Objectives'].value_counts().sort_index())

total_fin = pass2_flat['Number_Financial'].sum()
total_sus = pass2_flat['Number_Sustainable'].sum()
total_obj = pass2_flat['Number_of_Objectives'].sum()
print(f"\nObjective type breakdown:")
print(f"  Financial:   {total_fin} ({total_fin/total_obj*100:.1f}%)" if total_obj > 0 else "  Financial:   0")
print(f"  Sustainable: {total_sus} ({total_sus/total_obj*100:.1f}%)" if total_obj > 0 else "  Sustainable: 0")
print(f"  Funds with ≥1 sustainable objective: {(pass2_flat['Number_Sustainable'] > 0).sum()}")

PASS 2 SUMMARY:
  Funds: 22
  With ≥1 objective: 22
  Avg objectives: 2.2

Objective count distribution:
Number_of_Objectives
1    8
2    7
3    4
4    1
5    1
6    1
Name: count, dtype: int64

Objective type breakdown:
  Financial:   33 (67.3%)
  Sustainable: 9 (18.4%)
  Funds with ≥1 sustainable objective: 7


In [77]:
# === SAVE PASS 2 OUTPUT ===
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M")

# Save flattened (human-readable) version
p2_flat_filename = f'Pass2_Consolidated_{len(pass2_flat)}_funds_{timestamp}.xlsx'
p2_flat_path = os.path.join(OUTPUT_DIR, p2_flat_filename)
pass2_flat.to_excel(p2_flat_path, index=False, engine='openpyxl')

# Save raw JSON version (for Pass 3 input)
p2_raw_df = pass2_df.copy()
p2_raw_df['pass2_raw'] = p2_raw_df['pass2_raw'].apply(json.dumps)
p2_raw_filename = f'Pass2_Raw_{len(pass2_df)}_funds_{timestamp}.xlsx'
p2_raw_path = os.path.join(OUTPUT_DIR, p2_raw_filename)
p2_raw_df.to_excel(p2_raw_path, index=False, engine='openpyxl')

print(f"Saved flattened: {p2_flat_filename}")
print(f"Saved raw JSON:  {p2_raw_filename}")
print(f"  → Use the raw JSON file as input to Pass 3")

Saved flattened: Pass2_Consolidated_22_funds_20260608_1525.xlsx
Saved raw JSON:  Pass2_Raw_22_funds_20260608_1525.xlsx
  → Use the raw JSON file as input to Pass 3
